# Accessing data

In [2]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity


In [3]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    Meadowbank    15      22420    56948        0.825       0.023       0.015               0

## Holiday Function

In [4]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

In [5]:
# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Relative Ranking with 2 year intervals
- using similar code as relative_ranking.ipynb but using a 2 yearly interval (122 day) sample and plotting

In [18]:
# Build 2-year intervals (forward looking)

import os
import pandas as pd
import matplotlib.pyplot as plt

def build_two_year_windows(start=2004, end=2017):
    """
    Produces forward‑looking 2-year windows:
    For year Y, use Y and Y+1.
    Example: 2004 → (2004, 2005)
    """
    return [(y, y + 1) for y in range(start, end)]

In [22]:
# Cmpute relative ranking 122 days (2-year window)

def holiday_relative_rank_two_year_window_temp(
    demand, obs, info, station, years, holiday_name, holiday_lib,
    window_days=30
):
    """
    For each year Y:
        • Get holiday(Y) and holiday(Y+1)
        • Build ±window_days windows around both
        • Combine into a 122-day comparison pool
        • Compute relative rank of holiday(Y) demand against all 122 values
        • Produce ONE curve per year (no duplication)
    """

    demand.index = pd.to_datetime(demand.index)
    obs.index = pd.to_datetime(obs.index)
    obs = obs.select_dtypes(include="number")

    hourly = demand[[station]].resample("h").mean()
    obs_hourly = obs.resample("h").mean()

    curves = []
    temps = []
    labels = []

    for year in years:

        # Need next year's holiday for the 2-year window
        if (year + 1) not in years:
            continue

        if holiday_name not in holiday_lib:
            raise ValueError(f"Holiday '{holiday_name}' not found in holiday library")

        ref_this = holiday_lib[holiday_name](year)
        ref_next = holiday_lib[holiday_name](year + 1)

        # Build windows for both years
        start = ref_this - pd.Timedelta(days=window_days)
        end   = ref_next + pd.Timedelta(days=window_days)

        window = hourly.loc[start:end].copy()
        if window.empty:
            continue

        window["date"] = window.index.date
        window["hour"] = window.index.hour

        # Relative rank across the full 122-day pool
        window["rank"] = window.groupby("hour")[station].rank(method="average")
        n_days = window.groupby("hour")["date"].transform("nunique")
        window["relative_rank"] = window["rank"] / n_days

        # Extract holiday(Y) curve only
        expected_hours = pd.date_range(ref_this, ref_this + pd.Timedelta(hours=23), freq="h")
        holiday_day = window["relative_rank"].reindex(expected_hours)

        if holiday_day.isna().all():
            continue

        holiday_day.index = range(24)
        curves.append(holiday_day)
        labels.append(year)

        # Mean temperature for holiday(Y)
        temp_day = obs_hourly["t2m"].reindex(expected_hours)
        temps.append(temp_day.mean())

    # Convert to DataFrame
    import numpy as np
    import matplotlib

    curves_df = pd.DataFrame(curves, index=labels)
    temps = np.array(temps)

    norm = matplotlib.colors.Normalize(vmin=temps.min(), vmax=temps.max())
    cmap = matplotlib.colormaps.get_cmap("Spectral_r")

    fig, ax = plt.subplots(figsize=(14, 4))

    for i, (year, row) in enumerate(curves_df.iterrows()):
        color = cmap(norm(temps[i]))
        ax.plot(
            row.index,
            row.values,
            color=color,
            linewidth=2,
            alpha=0.9,
            label=f"{year}-{year+1} ({temps[i]:.1f}°C)"
        )

    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_xticks(range(24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(
        f"{full_name} {holiday_name} Relative Rank (2-year window, ±{window_days}d)\n"
        "Coloured by Mean Temperature"
    )
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Relative Rank")

    sm = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label(f"Mean {holiday_name} Temperature (°C)")

    ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.25))

    return fig

In [20]:
# Loop through all holidays and 4 substations (BLAKE, PUNCH, MEADO, MOSMA)

def generate_all_relative_rank_plots_two_year(
    demand, obs, info, holiday_lib,
    base_dir="/home/565/pv3484/aus_substation_electricity/figures/relative_ranking_2yr"
):
    """
    Generates one curve per year using a forward‑looking 2‑year window.
    Substations: BLAKE, PUNCH, MEADO, MOSMA.
    """

    years = list(range(2004, 2018))  # inclusive 2004–2017

    TARGET_STATIONS = ["BLAKE", "PUNCH", "MEADO", "MOSMA"]

    for holiday_name in holiday_lib.keys():

        holiday_folder = os.path.join(base_dir, holiday_name.replace(" ", "_"))
        os.makedirs(holiday_folder, exist_ok=True)

        print(f"\n=== {holiday_name} ===")

        for station in TARGET_STATIONS:

            print(f"   → {station}")

            fig = holiday_relative_rank_two_year_window_temp(
                demand=demand,
                obs=obs,
                info=info,
                station=station,
                years=years,
                holiday_name=holiday_name,
                holiday_lib=holiday_lib,
                window_days=30
            )

            out_path = os.path.join(holiday_folder, f"{station}.png")
            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)

In [23]:
# Run

generate_all_relative_rank_plots_two_year(
    demand=demand,
    obs=obs,
    info=info,
    holiday_lib=HOLIDAYS_VIC
)



=== New Year's Day ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA

=== Australia Day ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA

=== Good Friday ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA

=== Easter Saturday ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA

=== Easter Sunday ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA

=== Easter Monday ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA

=== ANZAC Day ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA

=== Christmas Day ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA

=== Boxing Day ===
   → BLAKE
   → PUNCH
   → MEADO
   → MOSMA


# Relative Ranking with 3 year intervals
- same as before, but increasing the intervals to three years (186 days)